<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%9E%D0%B1%D1%83%D1%87%D0%B0%D1%8E%D1%89%D0%B8%D0%B9_%D0%BD%D0%B0%D0%B1%D0%BE%D1%80__%D0%A7%D0%B0%D1%81%D1%82%D1%8C_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Цель работы:**  
Разработать модель, способную предсказывать состав следующего заказа пользователя на основе анализа его истории покупок. Это позволит повысить персонализацию сервиса и улучшить пользовательский опыт, а также оптимизировать процессы формирования корзины и планирования закупок.


# Введение и постановка задачи

Современные сервисы доставки продуктов активно используют данные о покупках пользователей для улучшения качества и персонализации сервиса. Анализ истории заказов позволяет прогнозировать, какие категории товаров пользователь может захотеть приобрести в следующий раз.

Целью данного проекта является создание модели, способной по истории заказов прогнозировать состав будущего заказа пользователя. Такая рекомендация поможет клиенту сэкономить время на формировании корзины, избежать забытых позиций и повысить удобство планирования закупок.

Задача формализована как многоклассовая классификация: необходимо предсказать для каждой пары (пользователь, категория), будет ли категория включена в следующий заказ. Для оценки качества модели используется метрика F1-score, которая учитывает баланс между точностью и полнотой предсказаний.

Данные и постановка задачи основаны на открытом соревновании в области электронных продаж.


# Описание набора данных

В проекте используется история заказов 20 000 пользователей, разделённая на тренировочную и тестовую выборки по дате. Тестовая выборка содержит заказы после определённой даты отсечки.

Основной тренировочный файл содержит следующие данные:  
- **user_id** — уникальный идентификатор пользователя  
- **order_completed_at** — дата и время завершения заказа  
- **cart** — категория товара, входящего в заказ (уникальные категории)

Задача — для каждой пары (пользователь, категория), встречающейся в тестовой выборке, предсказать бинарный признак: будет ли категория присутствовать в следующем заказе пользователя.

Идентификаторы пар представлены в формате `"{user_id};{category_id}"`, что учитывается при обработке данных.

Данные подготовлены на основе истории заказов с учётом особенностей временного разделения выборок.


In [226]:
# Подключение Google Drive для доступа к данным и сохранения результатов
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [227]:
# Создание структуры проекта
import os
from pathlib import Path
import pandas as pd
import numpy as np


# Основная папка проекта (как вы указали)
project_path = "/content/drive/MyDrive/Colab Notebooks/sm"

# Чек-лист папок и файлов
structure = [
    "data/raw",          # Исходные данные
    "data/processed",    # Очищенные данные
    "notebooks",         # Jupyter-тетради
    "src/models",        # Код моделей
    "src/utils",         # Утилиты
    "scripts",           # Скрипты обработки
]

# Создаём все директории
for folder in structure:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)


In [228]:
# Функция для рекурсивного вывода структуры папок проекта
def print_tree(root, prefix=""):
    files = sorted(os.listdir(root))
    for i, name in enumerate(files):
        path = os.path.join(root, name)
        is_last = (i == len(files) - 1)
        branch = "└── " if is_last else "├── "
        print(prefix + branch + name + ("/" if os.path.isdir(path) else ""))
        if os.path.isdir(path):
            new_prefix = prefix + ("    " if is_last else "│   ")
            print_tree(path, new_prefix)

print("\n=== Структура проекта (project_path) ===")
print_tree(project_path)


=== Структура проекта (project_path) ===
├── data/
│   ├── processed/
│   │   └── full_orders.parquet
│   └── raw/
│       ├── sample_submission.csv
│       └── train.csv
├── notebooks/
├── plan/
├── scripts/
└── src/
    ├── models/
    └── utils/


In [229]:
# Блок: Загрузка и первичный анализ train.csv
import pandas as pd
import os


train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/sm/data/raw/train.csv")

print("Структура и информация о train.csv:")
print(train.info())

Структура и информация о train.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3123064 entries, 0 to 3123063
Data columns (total 3 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   user_id             int64 
 1   order_completed_at  object
 2   cart                int64 
dtypes: int64(2), object(1)
memory usage: 71.5+ MB
None


In [230]:
train

,user_id,order_completed_at,cart
0,2,2015-03-22 09:25:46,399
1,2,2015-03-22 09:25:46,14
2,2,2015-03-22 09:25:46,198
3,2,2015-03-22 09:25:46,88
4,2,2015-03-22 09:25:46,157
...,...,...,...
3123059,12702,2020-09-03 23:45:45,441
3123060,12702,2020-09-03 23:45:45,92
3123061,12702,2020-09-03 23:45:45,431
3123062,12702,2020-09-03 23:45:45,24


In [231]:
print(f"Уникальных пользователей: {train['user_id'].nunique()}")

Уникальных пользователей: 20000


In [232]:
# Блок: Дополнительная статистика по train.csv

print("\nДополнительная статистика по train.csv:")

orders_per_user = train['user_id'].value_counts()
print("\nРаспределение количества заказов на пользователя:")
print(orders_per_user.describe())

print(f"\nУникальных категорий (корзин): {train['cart'].nunique()}")

train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])
print("\nСтатистика по датам заказов:")
print(f"Период с {train['order_completed_at'].min()} по {train['order_completed_at'].max()}")



Дополнительная статистика по train.csv:

Распределение количества заказов на пользователя:
count    20000.000000
mean       156.153200
std        200.840781
min          3.000000
25%         48.000000
50%         88.000000
75%        181.000000
max       3508.000000
Name: count, dtype: float64

Уникальных категорий (корзин): 881

Статистика по датам заказов:
Период с 2015-03-22 09:25:46 по 2020-09-03 23:45:45


In [233]:
# Преобразуем строку в формат даты
train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])

# Обрезаем дату, убирая время
train['order_completed_at'] = train['order_completed_at'].dt.floor('D')

# Проверим, как выглядит дата после обрезки
print(train['order_completed_at'].head())

0   2015-03-22
1   2015-03-22
2   2015-03-22
3   2015-03-22
4   2015-03-22
Name: order_completed_at, dtype: datetime64[ns]


In [234]:
# Установим стартовую дату
last_date = pd.to_datetime('2020-09-03')
interval_length_days = 30

# Проверим периоды, двигаясь назад
current_date = last_date
is_stable = False

while current_date >= train['order_completed_at'].min():
    # Рассчитаем предыдущие 30 дней
    prev_date = current_date - pd.Timedelta(interval_length_days, unit='D')

    # Отфильтруем заказы в рассматриваемом диапазоне
    period_orders = train[
        (train['order_completed_at'] >= prev_date) &
        (train['order_completed_at'] <= current_date)
    ]

    # Сформируем дни в этом диапазоне
    days_in_range = pd.date_range(start=prev_date, end=current_date)

    # Проверим наличие заказов в КАЖДЫЙ день периода
    missing_days = set(days_in_range) - set(period_orders['order_completed_at'].dt.normalize())

    # Количество заказов в периоде
    total_orders = len(period_orders)

    # Выводим результат
    if len(missing_days) == 0:
        print(f"Период с {prev_date} по {current_date} стабилен (каждый день есть заказы) - {total_orders} заказов")
    else:
        print(f"Период с {prev_date} по {current_date} нестабилен ({len(missing_days)} пропусков) - {total_orders} заказов")
        break

    # Передвинем дату назад
    current_date = prev_date

Период с 2020-08-04 00:00:00 по 2020-09-03 00:00:00 стабилен (каждый день есть заказы) - 464685 заказов
Период с 2020-07-05 00:00:00 по 2020-08-04 00:00:00 стабилен (каждый день есть заказы) - 472867 заказов
Период с 2020-06-05 00:00:00 по 2020-07-05 00:00:00 стабилен (каждый день есть заказы) - 454896 заказов
Период с 2020-05-06 00:00:00 по 2020-06-05 00:00:00 стабилен (каждый день есть заказы) - 357086 заказов
Период с 2020-04-06 00:00:00 по 2020-05-06 00:00:00 стабилен (каждый день есть заказы) - 293168 заказов
Период с 2020-03-07 00:00:00 по 2020-04-06 00:00:00 стабилен (каждый день есть заказы) - 200635 заказов
Период с 2020-02-06 00:00:00 по 2020-03-07 00:00:00 стабилен (каждый день есть заказы) - 154535 заказов
Период с 2020-01-07 00:00:00 по 2020-02-06 00:00:00 стабилен (каждый день есть заказы) - 135627 заказов
Период с 2019-12-08 00:00:00 по 2020-01-07 00:00:00 стабилен (каждый день есть заказы) - 138539 заказов
Период с 2019-11-08 00:00:00 по 2019-12-08 00:00:00 стабилен (ка

In [235]:
# Установим границы периода
start_date = pd.to_datetime('2019-09-09')
end_date = pd.to_datetime('2020-09-03')

# Фильтруем только те заказы, которые попадают в указанный период
train = train[
    (train['order_completed_at'] >= start_date) &
    (train['order_completed_at'] <= end_date)
]

In [236]:
train

,user_id,order_completed_at,cart
167611,2522,2019-09-09,798
167612,2522,2019-09-09,92
167613,2522,2019-09-09,19
167614,2522,2019-09-09,382
167615,2522,2019-09-09,22
...,...,...,...
3123059,12702,2020-09-03,441
3123060,12702,2020-09-03,92
3123061,12702,2020-09-03,431
3123062,12702,2020-09-03,24


In [237]:
# Установка границ периода
start_date = pd.to_datetime('2019-08-10')
end_date = pd.to_datetime('2020-09-03')

# Фильтрация данных
train = train[
    (train['order_completed_at'] >= start_date) &
    (train['order_completed_at'] <= end_date)
]

# Подсчёт статистики по пользователям
users_stats = train['user_id'].value_counts()

# Количестве уникальных пользователей
unique_users = train['user_id'].nunique()

# Количество уникальных корзин
unique_carts = train['cart'].nunique()

# Статистика по датам
start_order_date = train['order_completed_at'].min()
end_order_date = train['order_completed_at'].max()

# Выводим статистику
print("Дополнительная статистика:")
print("Распределение количества заказов на пользователя:")
print(users_stats.describe())

# Теперь правильное количество уникальных пользователей
print("\nУникальных пользователей:", unique_users)

# Количество уникальных корзин
print("\nУникальных категорий (корзин):", unique_carts)

# Статистика по датам
print("\nСтатистика по датам заказов:")
print(f"Период с {start_order_date} по {end_order_date}")

Дополнительная статистика:
Распределение количества заказов на пользователя:
count    20000.000000
mean       147.772650
std        180.092583
min          1.000000
25%         47.000000
50%         85.000000
75%        175.000000
max       2566.000000
Name: count, dtype: float64

Уникальных пользователей: 20000

Уникальных категорий (корзин): 870

Статистика по датам заказов:
Период с 2019-09-09 00:00:00 по 2020-09-03 00:00:00


In [238]:
# Блок: Загрузка и первичный анализ sample_submission.csv
sub = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/sm/data/raw/sample_submission.csv")

print("Структура и информация:")
print(sub.info())

Структура и информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 790449 entries, 0 to 790448
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   id      790449 non-null  object
 1   target  790449 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 12.1+ MB
None


In [239]:
sub

,id,target
0,0;133,0
1,0;5,1
2,0;10,0
3,0;396,1
4,0;14,0
...,...,...
790444,19998;26,0
790445,19998;31,0
790446,19998;29,1
790447,19998;798,1


In [240]:
# Блок: Агрегация корзин и добавление порядкового номера заказа
#
# Назначение:
#   - Сгруппировать данные по пользователям и времени завершения заказа.
#   - Собрать список категорий товаров (корзину) в каждом заказе.
#   - Добавить порядковый номер заказа для каждого пользователя.
#
# Вход:
#   - DataFrame train с данными заказов.
# Выход:
#   - DataFrame orders_agg с агрегированными корзинами и порядковыми номерами.
#
train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])

orders_agg = (
    train
    .groupby(['user_id', 'order_completed_at'])['cart']
    .apply(list)
    .reset_index()
    .rename(columns={'cart': 'cart_list'})
    .sort_values(['user_id', 'order_completed_at'])
)

orders_agg['order_number'] = (
    orders_agg
    .groupby('user_id')
    .cumcount() + 1
)

user_last_order = (
    orders_agg.groupby('user_id')['order_number'].max()
    .reset_index()
    .rename(columns={'order_number': 'max_order_number'})
)
orders_agg = orders_agg.merge(user_last_order, on='user_id', how='left')

full_orders = orders_agg.copy()

def unique_categories(df):
    return len(set(cat for cl in df['cart_list'] for cat in cl))

print("\nПолная выборка (full):")
print("  - Пользователей:", full_orders['user_id'].nunique())
print("  - Категорий:", unique_categories(full_orders))

print(full_orders.head(10).to_markdown(index=False))



Полная выборка (full):
  - Пользователей: 20000
  - Категорий: 870
|   user_id | order_completed_at   | cart_list                                                                                                         |   order_number |   max_order_number |
|----------:|:---------------------|:------------------------------------------------------------------------------------------------------------------|---------------:|-------------------:|
|         0 | 2020-07-19 00:00:00  | [20, 82, 441, 57, 14, 405, 430, 379]                                                                              |              1 |                  3 |
|         0 | 2020-08-24 00:00:00  | [133, 5, 26, 10, 382, 14, 22, 41, 25, 441, 411, 799, 432, 84, 83, 383, 409, 821, 405, 402, 57, 396, 379, 82, 157] |              2 |                  3 |
|         0 | 2020-09-02 00:00:00  | [803, 170, 84, 61, 440, 57, 55, 401, 398, 399, 169]                                                               |              3 

In [241]:
# Границы дат без времени (чистые даты)
last_date = full_orders['order_completed_at'].max().floor('D')
first_date = full_orders['order_completed_at'].min().floor('D')

# Подсчет разницы в днях
diff_in_days = (last_date - first_date).days

# Подсчет полного количества интервалов (целых частей по 30 дней)
num_full_intervals = diff_in_days // 30

# Остаток дней после целого деления
remaining_days = diff_in_days % 30

# Распечатываем результат
print("Максимальная дата:", last_date)
print("Минимальная дата:", first_date)
print("Всего дней между датами:", diff_in_days)
print("Количество полных интервалов по 30 дней:", num_full_intervals)
print("Оставшиеся дни после деления:", remaining_days)

Максимальная дата: 2020-09-03 00:00:00
Минимальная дата: 2019-09-09 00:00:00
Всего дней между датами: 360
Количество полных интервалов по 30 дней: 12
Оставшиеся дни после деления: 0


In [246]:
# Зафиксированная максимальная дата
last_date = full_orders['order_completed_at'].max().floor('D')

# Формируем временные интервалы вручную
current_date = last_date
intervals = []

# Заполняем полный диапазон интервалов
for _ in range(num_full_intervals):
    prev_date = current_date - pd.Timedelta(30, unit='D')
    intervals.append(pd.Interval(prev_date, current_date, closed='right'))
    current_date = prev_date

# Отсортируем интервалы от старых к новым
intervals.reverse()

# Выведем первые 12 интервалов
print("Первые 12 интервалов:")
for interval in intervals[:12]:
    print(interval)

Первые 12 интервалов:
(2019-09-09 00:00:00, 2019-10-09 00:00:00]
(2019-10-09 00:00:00, 2019-11-08 00:00:00]
(2019-11-08 00:00:00, 2019-12-08 00:00:00]
(2019-12-08 00:00:00, 2020-01-07 00:00:00]
(2020-01-07 00:00:00, 2020-02-06 00:00:00]
(2020-02-06 00:00:00, 2020-03-07 00:00:00]
(2020-03-07 00:00:00, 2020-04-06 00:00:00]
(2020-04-06 00:00:00, 2020-05-06 00:00:00]
(2020-05-06 00:00:00, 2020-06-05 00:00:00]
(2020-06-05 00:00:00, 2020-07-05 00:00:00]
(2020-07-05 00:00:00, 2020-08-04 00:00:00]
(2020-08-04 00:00:00, 2020-09-03 00:00:00]


In [256]:
# Назначаем интервалы каждой записи
full_orders['custom_interval'] = pd.cut(full_orders['order_completed_at'], bins=intervals, include_lowest=True, labels=range(len(intervals)), ordered=False)

In [255]:
full_orders

,user_id,order_completed_at,cart_list,order_number,max_order_number,custom_interval,interval_index
0,0,2020-07-19,"[20, 82, 441, 57, 14, 405, 430, 379]",1,3,"(2020-07-05 00:00:00, 2020-08-04 00:00:00]",10
1,0,2020-08-24,"[133, 5, 26, 10, 382, 14, 22, 41, 25, 441, 411...",2,3,"(2020-08-04 00:00:00, 2020-09-03 00:00:00]",11
2,0,2020-09-02,"[803, 170, 84, 61, 440, 57, 55, 401, 398, 399,...",3,3,"(2020-08-04 00:00:00, 2020-09-03 00:00:00]",11
3,1,2020-01-17,"[82, 798, 86, 421, 204, 55]",1,8,"(2020-01-07 00:00:00, 2020-02-06 00:00:00]",4
4,1,2020-02-06,[55],2,8,"(2020-01-07 00:00:00, 2020-02-06 00:00:00]",4
...,...,...,...,...,...,...,...
194079,19997,2020-08-31,"[0, 244, 49, 243, 239, 131, 420, 232, 231, 55,...",2,2,"(2020-08-04 00:00:00, 2020-09-03 00:00:00]",11
194080,19998,2020-08-30,"[6, 420, 398, 57, 415, 31, 26, 29]",1,3,"(2020-08-04 00:00:00, 2020-09-03 00:00:00]",11
194081,19998,2020-09-01,"[398, 57, 84, 61, 415, 6, 420]",2,3,"(2020-08-04 00:00:00, 2020-09-03 00:00:00]",11
194082,19998,2020-09-02,"[84, 798, 409, 19]",3,3,"(2020-08-04 00:00:00, 2020-09-03 00:00:00]",11


In [252]:
# Группируем по полю custom_interval
grouped_by_interval = full_orders.groupby('custom_interval')

# Показываем количество записей в каждом интервале
print(grouped_by_interval.size())

custom_interval
(2019-09-09 00:00:00, 2019-10-09 00:00:00]     3721
(2019-10-09 00:00:00, 2019-11-08 00:00:00]     6896
(2019-11-08 00:00:00, 2019-12-08 00:00:00]     9310
(2019-12-08 00:00:00, 2020-01-07 00:00:00]     8467
(2020-01-07 00:00:00, 2020-02-06 00:00:00]     8695
(2020-02-06 00:00:00, 2020-03-07 00:00:00]     9819
(2020-03-07 00:00:00, 2020-04-06 00:00:00]    12483
(2020-04-06 00:00:00, 2020-05-06 00:00:00]    18169
(2020-05-06 00:00:00, 2020-06-05 00:00:00]    22793
(2020-06-05 00:00:00, 2020-07-05 00:00:00]    30996
(2020-07-05 00:00:00, 2020-08-04 00:00:00]    31550
(2020-08-04 00:00:00, 2020-09-03 00:00:00]    31129
dtype: int64


/tmp/ipython-input-1690598797.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_by_interval = full_orders.groupby('custom_interval')


In [254]:
# Преобразуем интервалы в числовые индексы
full_orders['interval_index'] = pd.Categorical(full_orders['custom_interval']).codes

# Сортируем по индексу интервалов
sorted_full_orders = full_orders.sort_values(by='interval_index')

# Проверяем результат
print(sorted_full_orders[['custom_interval', 'interval_index']].head())

      custom_interval  interval_index
41144             NaN              -1
21430             NaN              -1
2392              NaN              -1
28317             NaN              -1
20772             NaN              -1


In [259]:
# Проверяем наличие пустых значений в колонке custom_interval
missing_values = full_orders['custom_interval'].isnull()

# Количество строк с пустыми значениями
print("Количество строк с пустыми значениями в custom_interval:", missing_values.sum())

# Просмотр первых пустых строк
print("Первые строки с пустыми значениями:")
full_orders[missing_values].head()

Количество строк с пустыми значениями в custom_interval: 56
Первые строки с пустыми значениями:


,user_id,order_completed_at,cart_list,order_number,max_order_number,custom_interval,interval_index
425,40,2019-09-09,"[405, 85, 48, 393, 395, 396, 398, 400, 61, 382...",1,34,NaN,-1
725,75,2019-09-09,"[19, 25, 383, 382, 5, 9, 84, 379, 85]",1,104,NaN,-1
2392,194,2019-09-09,"[23, 27, 383, 14, 384]",1,11,NaN,-1
2772,233,2019-09-09,"[712, 198, 104, 5, 100, 99, 98, 435, 0, 42, 41...",1,4,NaN,-1
2947,260,2019-09-09,"[443, 440, 437, 14, 15, 16, 21, 396, 400, 433,...",1,77,NaN,-1


In [258]:
missing_values

,custom_interval
0,False
1,False
2,False
3,False
4,False
...,...
194079,False
194080,False
194081,False
194082,False


In [224]:
# Проверим первую и последнюю даты
test_dates = ["2019-09-09", "2020-09-03"]

for date_str in test_dates:
    date_obj = pd.to_datetime(date_str)
    row = full_orders[full_orders['order_completed_at'] == date_obj]
    if not row.empty:
        print(f"Дата {date_str}: Интервал {row['custom_interval'].values[0]}")
    else:
        print(f"Дата {date_str} отсутствует в данных!")

Дата 2019-09-09: Интервал 0
Дата 2020-09-03: Интервал 12


In [203]:
# Получим границы нулевого и 66-го интервалов
zero_interval = intervals[0]
sixty_six_interval = intervals[-1]

# Выведем границы интервалов
print("Границы нулевого интервала:", zero_interval.left, "-", zero_interval.right)
print("Границы 66-го интервала:", sixty_six_interval.left, "-", sixty_six_interval.right)

Границы нулевого интервала: 2019-08-10 00:00:00 - 2019-09-09 00:00:00
Границы 66-го интервала: 2020-08-04 00:00:00 - 2020-09-03 00:00:00


In [204]:
# Сортировка записей по дате
sorted_df = full_orders.sort_values(by='order_completed_at', ascending=True)

# Промежуточная проверка первых записей
print("Первые 5 записей после сортировки по дате:")
print(sorted_df[['user_id', 'order_completed_at', 'custom_interval']].head(5).to_markdown(index=False))

# Промежуточная проверка последних записей
print("Последние 5 записей после сортировки по дате:")
print(sorted_df[['user_id', 'order_completed_at', 'custom_interval']].tail(5).to_markdown(index=False))

Первые 5 записей после сортировки по дате:
|   user_id | order_completed_at   |   custom_interval |
|----------:|:---------------------|------------------:|
|      2356 | 2019-09-09 00:00:00  |                 0 |
|      2434 | 2019-09-09 00:00:00  |                 0 |
|      2321 | 2019-09-09 00:00:00  |                 0 |
|      2269 | 2019-09-09 00:00:00  |                 0 |
|      2228 | 2019-09-09 00:00:00  |                 0 |
Последние 5 записей после сортировки по дате:
|   user_id | order_completed_at   |   custom_interval |
|----------:|:---------------------|------------------:|
|     13466 | 2020-09-03 00:00:00  |                12 |
|     12970 | 2020-09-03 00:00:00  |                12 |
|     19345 | 2020-09-03 00:00:00  |                12 |
|     19719 | 2020-09-03 00:00:00  |                12 |
|      7364 | 2020-09-03 00:00:00  |                12 |


In [205]:
full_orders

,user_id,order_completed_at,cart_list,order_number,max_order_number,custom_interval
0,0,2020-07-19,"[20, 82, 441, 57, 14, 405, 430, 379]",1,3,11
1,0,2020-08-24,"[133, 5, 26, 10, 382, 14, 22, 41, 25, 441, 411...",2,3,12
2,0,2020-09-02,"[803, 170, 84, 61, 440, 57, 55, 401, 398, 399,...",3,3,12
3,1,2020-01-17,"[82, 798, 86, 421, 204, 55]",1,8,5
4,1,2020-02-06,[55],2,8,5
...,...,...,...,...,...,...
194079,19997,2020-08-31,"[0, 244, 49, 243, 239, 131, 420, 232, 231, 55,...",2,2,12
194080,19998,2020-08-30,"[6, 420, 398, 57, 415, 31, 26, 29]",1,3,12
194081,19998,2020-09-01,"[398, 57, 84, 61, 415, 6, 420]",2,3,12
194082,19998,2020-09-02,"[84, 798, 409, 19]",3,3,12


In [206]:
# Блок: Добавление даты без времени и глобального номера заказа
#
# Назначение:
#   - Преобразовать дату и время заказа к дате без времени.
#   - Создать глобальный порядковый номер заказа по дате для всех пользователей.
#
# Вход:
#   - DataFrame full_orders с историей заказов.
# Выход:
#   - DataFrame full_orders с добавленными колонками order_date и global_order_number.
#
full_orders['order_date'] = full_orders['order_completed_at'].dt.date
full_orders = full_orders.sort_values('order_completed_at')

unique_dates = sorted(full_orders['order_date'].unique())
date_to_global_order_num = {date: idx + 1 for idx, date in enumerate(unique_dates)}
full_orders['global_order_number'] = full_orders['order_date'].map(date_to_global_order_num)


In [207]:
print(full_orders.head().to_markdown(index=False))

|   user_id | order_completed_at   | cart_list                                                                                                                                                                                           |   order_number |   max_order_number |   custom_interval | order_date   |   global_order_number |
|----------:|:---------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|---------------:|-------------------:|------------------:|:-------------|----------------------:|
|      2356 | 2019-09-09 00:00:00  | [421, 377, 376, 799, 420, 798, 392, 384, 383, 92, 89, 5]                                                                                                                                            |              1 |                  7 |                 0 | 2019-09-09   |                     1 |
|   

In [208]:
# Создаем колонку с годом и месяцем для каждого заказа
full_orders['year_month'] = full_orders['order_completed_at'].dt.to_period('M')

In [209]:
print(full_orders.head(10).to_markdown(index=False))

|   user_id | order_completed_at   | cart_list                                                                                                                                                                                           |   order_number |   max_order_number |   custom_interval | order_date   |   global_order_number | year_month   |
|----------:|:---------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|---------------:|-------------------:|------------------:|:-------------|----------------------:|:-------------|
|      2356 | 2019-09-09 00:00:00  | [421, 377, 376, 799, 420, 798, 392, 384, 383, 92, 89, 5]                                                                                                                                            |              1 |                  7 |                 0 | 2019-09-09   

In [210]:
# --- Индивидуальный порядковый номер месяца для каждого пользователя ---
# Получаем уникальные месяцы для каждого пользователя
unique_user_months = (
    full_orders[['user_id', 'year_month']]
    .drop_duplicates()
    .sort_values(['user_id', 'year_month'])
)

# Добавляем порядковый номер месяца для пользователя
unique_user_months['individual_month'] = unique_user_months.groupby('user_id').cumcount() + 1

# Объединяем обратно с исходным датафреймом
full_orders = full_orders.merge(unique_user_months, on=['user_id', 'year_month'], how='left')

In [211]:
print(full_orders.head(10).to_markdown(index=False))

|   user_id | order_completed_at   | cart_list                                                                                                                                                                                           |   order_number |   max_order_number |   custom_interval | order_date   |   global_order_number | year_month   |   individual_month |
|----------:|:---------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|---------------:|-------------------:|------------------:|:-------------|----------------------:|:-------------|-------------------:|
|      2356 | 2019-09-09 00:00:00  | [421, 377, 376, 799, 420, 798, 392, 384, 383, 92, 89, 5]                                                                                                                                            |              1 |             

In [212]:
# --- Глобальный порядковый номер месяца по всему датафрейму ---
# Получаем уникальные глобальные месяцы по дате (без учета пользователя)
unique_global_months = (
    full_orders[['year_month']]
    .drop_duplicates()
    .sort_values('year_month')
    .reset_index(drop=True)
)
unique_global_months['global_month'] = unique_global_months.index + 1

# Объединяем обратно с основным датафреймом
full_orders = full_orders.merge(unique_global_months, on='year_month', how='left')


In [213]:
print(full_orders.head(10).to_markdown(index=False))

|   user_id | order_completed_at   | cart_list                                                                                                                                                                                           |   order_number |   max_order_number |   custom_interval | order_date   |   global_order_number | year_month   |   individual_month |   global_month |
|----------:|:---------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|---------------:|-------------------:|------------------:|:-------------|----------------------:|:-------------|-------------------:|---------------:|
|      2356 | 2019-09-09 00:00:00  | [421, 377, 376, 799, 420, 798, 392, 384, 383, 92, 89, 5]                                                                                                                                         

In [214]:
full_orders

,user_id,order_completed_at,cart_list,order_number,max_order_number,custom_interval,order_date,global_order_number,year_month,individual_month,global_month
0,2356,2019-09-09,"[421, 377, 376, 799, 420, 798, 392, 384, 383, ...",1,7,0,2019-09-09,1,2019-09,1,1
1,2434,2019-09-09,"[807, 29, 798, 41, 144, 43, 384, 42, 376, 379,...",1,47,0,2019-09-09,1,2019-09,1,1
2,2321,2019-09-09,"[109, 0, 41, 26]",1,24,0,2019-09-09,1,2019-09,1,1
3,2269,2019-09-09,"[84, 77, 395, 420, 443, 383, 388, 439, 384, 23...",1,28,0,2019-09-09,1,2019-09,1,1
4,2228,2019-09-09,"[57, 403, 89, 379, 14, 61, 376, 55, 169, 26, 3...",1,2,0,2019-09-09,1,2019-09,1,1
...,...,...,...,...,...,...,...,...,...,...,...
194079,13466,2020-09-03,"[169, 57, 409, 0, 23, 99, 421, 16, 431, 61, 14...",23,23,12,2020-09-03,361,2020-09,5,13
194080,12970,2020-09-03,"[104, 55, 14]",8,8,12,2020-09-03,361,2020-09,5,13
194081,19345,2020-09-03,"[29, 55, 30, 808, 376, 61, 198, 425, 57, 388, ...",5,5,12,2020-09-03,361,2020-09,3,13
194082,19719,2020-09-03,"[172, 432, 173, 409, 10, 61, 397, 395, 388, 38...",4,4,12,2020-09-03,361,2020-09,2,13


In [156]:
# Парсим пары user_id и category_id из sample_submission.csv
sub['user_id'] = sub['id'].apply(lambda x: int(x.split(";")[0]))
sub['category_id'] = sub['id'].apply(lambda x: int(x.split(";")[1]))

In [20]:
sub

,id,target,user_id,category_id
0,0;133,0,0,133
1,0;5,1,0,5
2,0;10,0,0,10
3,0;396,1,0,396
4,0;14,0,0,14
...,...,...,...,...
790444,19998;26,0,19998,26
790445,19998;31,0,19998,31
790446,19998;29,1,19998,29
790447,19998;798,1,19998,798


In [21]:
# Приводим train.csv к удобному виду
train['user_id'] = train['user_id']
train['category_id'] = train['cart']


In [22]:
train

,user_id,order_completed_at,cart,category_id
0,2,2015-03-22 09:25:46,399,399
1,2,2015-03-22 09:25:46,14,14
2,2,2015-03-22 09:25:46,198,198
3,2,2015-03-22 09:25:46,88,88
4,2,2015-03-22 09:25:46,157,157
...,...,...,...,...
3123059,12702,2020-09-03 23:45:45,441,441
3123060,12702,2020-09-03 23:45:45,92,92
3123061,12702,2020-09-03 23:45:45,431,431
3123062,12702,2020-09-03 23:45:45,24,24


In [23]:
# Строим списки для проверки
pairs_in_train = set(zip(train['user_id'], train['category_id']))  # Все пары из train.csv
pairs_in_sample = list(zip(sub['user_id'], sub['category_id']))   # Все пары из sample_submission.csv

# Проверка наличия каждой пары в train.csv
pairs_existence = [pair in pairs_in_train for pair in pairs_in_sample]

# Итоговый отчет
num_pairs_in_sample = len(pairs_in_sample)                         # Всего пар в sample_submission.csv
num_pairs_in_train = len(pairs_in_train)                           # Всего пар в train.csv
num_found = sum(pairs_existence)                                   # Сколько пар найдено в train.csv
percent_found = (num_found / num_pairs_in_sample) * 100            # Процент нахождения пар

print(f"Всего пар в sample_submission.csv: {num_pairs_in_sample}")
print(f"Всего пар в train.csv: {num_pairs_in_train}")
print(f"Количество найденных пар: {num_found} ({percent_found:.2f}%)")

# Если есть отсутствие пар, показываем первые пять
missing_pairs = [pair for pair, exists in zip(pairs_in_sample, pairs_existence) if not exists]
if missing_pairs:
    print("\nПервые пять отсутствующих пар:")
    print(missing_pairs[:5])
else:
    print("\nВсе пары из sample_submission.csv успешно найдены в train.csv!")

Всего пар в sample_submission.csv: 790449
Всего пар в train.csv: 1117600
Количество найденных пар: 790449 (100.00%)

Все пары из sample_submission.csv успешно найдены в train.csv!


In [24]:
import pandas as pd
# Выделяем user_id из sample_submission.csv
sub['user_id'] = sub['id'].apply(lambda x: int(x.split(";")[0]))

# Перечень пользователей из sample_submission.csv
users_in_sub = set(sub['user_id'])

# Информация о пользователях в full_orders
user_stats = (
    full_orders
    .groupby('user_id')
    [['order_number', 'individual_month']]
    .agg({'order_number': ['min', 'max', 'mean'], 'individual_month': 'max'})
    .reset_index()
)

# Пересечение пользователей из sample_submission.csv и full_orders
stats_for_sub_users = user_stats[user_stats['user_id'].isin(users_in_sub)]

# Количество пользователей в sample_submission.csv
num_users_in_sub = len(users_in_sub)

# Количество пользователей в full_orders
num_users_in_full = len(user_stats)

# Количество пользователей из sample_submission.csv, представленных в full_orders
num_sub_users_in_full = stats_for_sub_users.shape[0]

# Итоговый отчет
print(f"Количество пользователей в sample_submission.csv: {num_users_in_sub}")
print(f"Количество пользователей в full_orders: {num_users_in_full}")
print(f"Пересечение пользователей (количество пользователей из sample_submission.csv, присутствующих в full_orders): {num_sub_users_in_full}")

# Другие важные статистики
avg_months_active = stats_for_sub_users['individual_month']['max'].mean()
avg_orders = stats_for_sub_users['order_number']['mean'].mean()
min_orders = stats_for_sub_users['order_number']['min'].min()
max_orders = stats_for_sub_users['order_number']['max'].max()

print(f"Средняя продолжительность пользования системой (месяцев): {avg_months_active:.2f}")
print(f"Среднее количество заказов: {avg_orders:.2f}")
print(f"Минимальное количество заказов: {min_orders}")
print(f"Максимальное количество заказов: {max_orders}")

Количество пользователей в sample_submission.csv: 13036
Количество пользователей в full_orders: 20000
Пересечение пользователей (количество пользователей из sample_submission.csv, присутствующих в full_orders): 13036
Средняя продолжительность пользования системой (месяцев): 5.12
Среднее количество заказов: 6.66
Минимальное количество заказов: 1
Максимальное количество заказов: 187


In [25]:
import pandas as pd

# Преобразование дат в нужный формат
full_orders['order_completed_at'] = pd.to_datetime(full_orders['order_completed_at'])

# Последняя неделя
last_week_start = full_orders['order_completed_at'].max() - pd.Timedelta(days=6)
last_week_end = full_orders['order_completed_at'].max()
week_orders = full_orders[(full_orders['order_completed_at'] >= last_week_start) &
                         (full_orders['order_completed_at'] <= last_week_end)]

# Последний месяц
last_month_start = full_orders['order_completed_at'].max() - pd.Timedelta(days=30)
month_orders = full_orders[(full_orders['order_completed_at'] >= last_month_start) &
                          (full_orders['order_completed_at'] <= full_orders['order_completed_at'].max())]

# Итоговый отчет
print(f"Статистика за последнюю неделю ({last_week_start.date()} - {last_week_end.date()}):")
print(f"Количество уникальных пользователей, сделавших заказы: {week_orders['user_id'].nunique()}")
print(f"Общее количество заказов: {len(week_orders)}")

print("\nСтатистика за последний месяц:")
print(f"Количество уникальных пользователей, сделавших заказы: {month_orders['user_id'].nunique()}")
print(f"Общее количество заказов: {len(month_orders)}")

Статистика за последнюю неделю (2020-08-28 - 2020-09-03):
Количество уникальных пользователей, сделавших заказы: 5361
Общее количество заказов: 6336

Статистика за последний месяц:
Количество уникальных пользователей, сделавших заказы: 13082
Общее количество заказов: 32199


Количество уникальных пользователей в последнем месяце: в последнем месяце активной была группа из 13082 пользователей, что близко к общему числу пользователей в файле sample_submission.csv (13036).

In [26]:
import pandas as pd

# Предполагая, что ваш объект full_orders уже готов
# Сохраняем в папку processed в формате Parquet
full_orders.to_parquet("/content/drive/MyDrive/Colab Notebooks/sm/data/processed/full_orders.parquet", index=False)

print("Файл успешно сохранён в формате Parquet в папку 'processed'.")

Файл успешно сохранён в формате Parquet в папку 'processed'.
